# Section 2: Electric Motor Fundamentals

## Learning Objectives

After completing this section, you will understand:

- **Motor Types**: Differences between IPM and FSCW permanent magnet motors
- **Control Strategies**: MTPA, flux weakening, and MTPV control methods
- **Performance Characteristics**: Efficiency maps, power factor maps, and operating envelopes
- **Design Parameters**: Geometric variables and their impact on motor performance
- **Operating Regions**: Understanding speed-torque characteristics and limitations

## Overview

This section provides the foundational knowledge of electric motors that forms the basis for performance map prediction. We focus on two main types of synchronous permanent magnet motors commonly used in electric vehicle applications.

In [ ]:
# Motor fundamentals visualization
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle, Circle, Polygon
import pandas as pd
from scipy.interpolate import griddata

print("⚡ Electric Motor Fundamentals")
print("=" * 50)
print("📚 Understanding motor types, control strategies, and performance characteristics")

## Motor Types and Geometries

This work focuses on two main types of synchronous permanent magnet motors:

### 1. Interior Permanent Magnet (IPM) Motor
- **Configuration**: 24 slots, 4 poles
- **Application**: High-performance EV applications
- **Features**: High torque density, good flux-weakening capability
- **Design parameters**: 12 geometric variables (X₁-X₁₂)

### 2. Fractional-Slot Concentrated Winding (FSCW) PM Motor
- **Configuration**: 12 slots, 10 poles
- **Application**: Compact EV applications
- **Features**: High efficiency, low cogging torque
- **Design parameters**: 10 geometric variables (X₁-X₁₀)

In [ ]:
# Motor geometry and characteristics visualization
def visualize_motor_types():
    """Visualize IPM and FSCW motor geometries and characteristics"""
    
    # Create motor geometry visualizations
    fig = plt.figure(figsize=(20, 12))
    
    # IPM Motor Geometry
    ax1 = plt.subplot(2, 4, 1)
    
    # Simplified IPM motor cross-section
    theta = np.linspace(0, 2*np.pi, 100)
    
    # Stator outline
    stator_outer = 100
    stator_inner = 60
    rotor_outer = 55
    
    # Draw stator
    ax1.plot(stator_outer * np.cos(theta), stator_outer * np.sin(theta), 'k-', linewidth=2)
    ax1.plot(stator_inner * np.cos(theta), stator_inner * np.sin(theta), 'k-', linewidth=2)
    
    # Draw rotor
    ax1.fill_between(rotor_outer * np.cos(theta), rotor_outer * np.sin(theta), 
                    color='lightgray', alpha=0.5)
    
    # Draw simplified magnets (V-shaped)
    magnet_angles = np.array([0, 45, 90, 135]) * np.pi / 180
    for angle in magnet_angles:
        # V-shaped magnet representation
        mag_x1 = [35 * np.cos(angle - 0.2), 25 * np.cos(angle), 35 * np.cos(angle + 0.2)]
        mag_y1 = [35 * np.sin(angle - 0.2), 25 * np.sin(angle), 35 * np.sin(angle + 0.2)]
        ax1.fill(mag_x1, mag_y1, color='red', alpha=0.7)
        
        mag_x2 = [-35 * np.cos(angle - 0.2), -25 * np.cos(angle), -35 * np.cos(angle + 0.2)]
        mag_y2 = [-35 * np.sin(angle - 0.2), -25 * np.sin(angle), -35 * np.sin(angle + 0.2)]
        ax1.fill(mag_x2, mag_y2, color='blue', alpha=0.7)
    
    # Draw slots
    n_slots = 24
    slot_angles = np.linspace(0, 2*np.pi, n_slots, endpoint=False)
    for angle in slot_angles:
        slot_x = [stator_inner * np.cos(angle), stator_outer * np.cos(angle)]
        slot_y = [stator_inner * np.sin(angle), stator_outer * np.sin(angle)]
        ax1.plot(slot_x, slot_y, 'k-', linewidth=1)
    
    ax1.set_xlim(-120, 120)
    ax1.set_ylim(-120, 120)
    ax1.set_aspect('equal')
    ax1.set_title('IPM Motor\n(24 slots, 4 poles)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('x (mm)')
    ax1.set_ylabel('y (mm)')
    ax1.grid(True, alpha=0.3)
    
    # FSCW Motor Geometry
    ax2 = plt.subplot(2, 4, 2)
    
    # Simplified FSCW motor cross-section
    stator_outer = 100
    stator_inner = 60
    rotor_outer = 55
    
    # Draw stator
    ax2.plot(stator_outer * np.cos(theta), stator_outer * np.sin(theta), 'k-', linewidth=2)
    ax2.plot(stator_inner * np.cos(theta), stator_inner * np.sin(theta), 'k-', linewidth=2)
    
    # Draw rotor
    ax2.fill_between(rotor_outer * np.cos(theta), rotor_outer * np.sin(theta), 
                    color='lightgray', alpha=0.5)
    
    # Draw surface magnets
    n_poles = 10
    pole_angles = np.linspace(0, 2*np.pi, n_poles, endpoint=False)
    for i, angle in enumerate(pole_angles):
        color = 'red' if i % 2 == 0 else 'blue'
        pole_x1 = rotor_outer * np.cos(angle - np.pi/n_poles/2)
        pole_y1 = rotor_outer * np.sin(angle - np.pi/n_poles/2)
        pole_x2 = rotor_outer * np.cos(angle + np.pi/n_poles/2)
        pole_y2 = rotor_outer * np.sin(angle + np.pi/n_poles/2)
        
        inner_x = 45 * np.cos(angle)
        inner_y = 45 * np.sin(angle)
        
        ax2.fill([pole_x1, pole_x2, inner_x], [pole_y1, pole_y2, inner_y], 
                color=color, alpha=0.7)
    
    # Draw concentrated slots
    n_slots = 12
    slot_angles = np.linspace(0, 2*np.pi, n_slots, endpoint=False)
    for angle in slot_angles:
        # Wider slot opening for concentrated winding
        slot_width = 0.15  # radians
        slot_x1 = [stator_inner * np.cos(angle - slot_width/2), 
                  stator_outer * np.cos(angle - slot_width/2)]
        slot_y1 = [stator_inner * np.sin(angle - slot_width/2), 
                  stator_outer * np.sin(angle - slot_width/2)]
        slot_x2 = [stator_inner * np.cos(angle + slot_width/2), 
                  stator_outer * np.cos(angle + slot_width/2)]
        slot_y2 = [stator_inner * np.sin(angle + slot_width/2), 
                  stator_outer * np.sin(angle + slot_width/2)]
        
        ax2.plot([slot_x1[0], slot_x2[0]], [slot_y1[0], slot_y2[0]], 'k-', linewidth=2)
        ax2.plot(slot_x1, slot_y1, 'k-', linewidth=1)
        ax2.plot(slot_x2, slot_y2, 'k-', linewidth=1)
    
    ax2.set_xlim(-120, 120)
    ax2.set_ylim(-120, 120)
    ax2.set_aspect('equal')
    ax2.set_title('FSCW PM Motor\n(12 slots, 10 poles)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('x (mm)')
    ax2.set_ylabel('y (mm)')
    ax2.grid(True, alpha=0.3)
    
    # Design Parameter Comparison
    ax3 = plt.subplot(2, 4, 3)
    
    # Create parameter comparison table as text
    parameter_data = [
        ['Parameter', 'IPM Motor', 'FSCW Motor'],
        ['Slots', '24', '12'],
        ['Poles', '4', '10'],
        ['Rated Current', '33.33 A', '300 A'],
        ['DC Voltage', '400 V', '450 V'],
        ['Design Vars', '12 (X₁-X₁₂)', '10 (X₁-X₁₀)']
    ]
    
    table_data = []
    for row in parameter_data[1:]:
        table_data.append(row)
    
    table = ax3.table(cellText=table_data,
                    colLabels=parameter_data[0],
                    cellLoc='center',
                    loc='center',
                    colWidths=[0.4, 0.3, 0.3])
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Style the table
    for i in range(len(parameter_data)):
        for j in range(len(parameter_data[0])):
            cell = table[(i, j)]
            if i == 0:  # Header row
                cell.set_facecolor('#4CAF50')
                cell.set_text_props(weight='bold', color='white')
            else:
                cell.set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')
    
    ax3.axis('off')
    ax3.set_title('Motor Specifications', fontsize=12, fontweight='bold')
    
    # Performance Characteristics Comparison
    ax4 = plt.subplot(2, 4, 4)
    
    performance_metrics = ['Torque\nDensity', 'Efficiency', 'Power\nFactor', 'Cogging\nTorque', 'Speed\nRange']
    ipm_scores = [8, 7, 8, 6, 9]
    fscw_scores = [7, 9, 7, 9, 8]
    
    x = np.arange(len(performance_metrics))
    width = 0.35
    
    bars1 = ax4.bar(x - width/2, ipm_scores, width, label='IPM', color='#3498DB', alpha=0.7)
    bars2 = ax4.bar(x + width/2, fscw_scores, width, label='FSCW', color='#E74C3C', alpha=0.7)
    
    ax4.set_xlabel('Performance Metrics', fontweight='bold')
    ax4.set_ylabel('Score (1-10)', fontweight='bold')
    ax4.set_title('Performance Comparison', fontsize=12, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(performance_metrics)
    ax4.legend()
    ax4.set_ylim(0, 10)
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Electric Motor Types and Characteristics', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print key insights
    print("⚡ Electric Motor Characteristics:")
    print("=" * 40)
    print(f"🔧 IPM Motor: {parameter_data[1][1]} slots, {parameter_data[2][1]} poles, {parameter_data[5][1]} design variables")
    print(f"🔧 FSCW Motor: {parameter_data[1][2]} slots, {parameter_data[2][2]} poles, {parameter_data[5][2]} design variables")
    print(f"📊 IPM Peak Torque Density Score: {max(ipm_scores)}/10")
    print(f"📊 FSCW Peak Efficiency Score: {max(fscw_scores)}/10")

visualize_motor_types()

## Performance Characteristics

### Control Strategies

Electric motors in EV applications require sophisticated control strategies to optimize performance across different operating conditions:

#### 1. Maximum Torque Per Ampere (MTPA)
- **Objective**: Maximize torque production per unit current
- **Region**: Below base speed
- **Benefits**: Optimal efficiency in low-speed region
- **Method**: Optimize d-q current angle for maximum torque

#### 2. Flux Weakening
- **Objective**: Extend speed range beyond base speed
- **Region**: Above base speed
- **Benefits**: Higher maximum speeds
- **Method**: Adjust d-axis current to reduce back-EMF

#### 3. Maximum Torque Per Volt (MTPV)
- **Objective**: Maximize torque under voltage constraints
- **Region**: High-speed operation
- **Benefits**: Optimal performance at voltage limits
- **Method**: Balance current and voltage constraints

In [ ]:
# Control strategy regions visualization
def visualize_control_strategies():
    """Visualize control strategy regions and operating envelopes"""
    
    # Create operating regions visualization
    speed = np.linspace(0, 6000, 100)
    torque = np.linspace(0, 250, 100)
    SPEED, TORQUE = np.meshgrid(speed, torque)
    
    # Define operating regions
    base_speed = 3000  # RPM
    
    # MTPA region (below base speed)
    mtpa_region = SPEED <= base_speed
    
    # Flux weakening region (above base speed)
    fw_region = SPEED > base_speed
    
    # Create operating region map
    region_map = np.zeros_like(SPEED)
    region_map[mtpa_region] = 1  # MTPA
    region_map[fw_region] = 2    # Flux weakening
    
    # Add torque limit envelope
    torque_limit = 200 * np.exp(-SPEED / 8000)  # Exponential torque drop
    for i in range(len(torque)):
        for j in range(len(speed)):
            if TORQUE[i, j] > torque_limit[j]:
                region_map[i, j] = 0  # Outside operating envelope
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Motor Control Strategies and Operating Regions', fontsize=16, fontweight='bold')
    
    # Plot 1: Control strategy regions
    ax1 = axes[0, 0]
    im1 = ax1.contourf(SPEED, TORQUE, region_map, levels=[0, 1, 2, 3], 
                     colors=['lightgray', 'lightblue', 'lightcoral'], alpha=0.7)
    ax1.set_xlabel('Speed (RPM)', fontweight='bold')
    ax1.set_ylabel('Torque (Nm)', fontweight='bold')
    ax1.set_title('Control Strategy Regions', fontweight='bold')
    
    # Add region labels
    ax1.text(1500, 200, 'MTPA Region', fontsize=12, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue', alpha=0.8))
    ax1.text(4500, 100, 'Flux Weakening\nRegion', fontsize=12, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor='lightcoral', alpha=0.8))
    ax1.text(3000, 30, f'Base Speed\n{base_speed} RPM', fontsize=10, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.8))
    
    # Draw torque limit line
    ax1.plot(speed, torque_limit, 'k-', linewidth=2, label='Torque Limit')
    ax1.axvline(x=base_speed, color='green', linestyle='--', linewidth=2, label='Base Speed')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Current trajectories
    ax2 = axes[0, 1]
    
    # Simulate current trajectories for different control strategies
    speeds_test = np.linspace(500, 5500, 50)
    
    # MTPA trajectory (constant current angle)
    mtpa_current = 150 * np.ones_like(speeds_test)
    mtpa_angle = 45 * np.ones_like(speeds_test)  # degrees
    
    # Flux weakening trajectory
    fw_current = np.where(speeds_test <= base_speed, 150, 150 * (base_speed / speeds_test))
    fw_angle = np.where(speeds_test <= base_speed, 45, 30)  # degrees
    
    ax2.plot(speeds_test, mtpa_current, 'b-', linewidth=2, label='MTPA Current')
    ax2.plot(speeds_test, fw_current, 'r-', linewidth=2, label='Flux Weakening Current')
    ax2.set_xlabel('Speed (RPM)', fontweight='bold')
    ax2.set_ylabel('Current Magnitude (A)', fontweight='bold')
    ax2.set_title('Current Magnitude Trajectories', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axvline(x=base_speed, color='green', linestyle='--', alpha=0.7)
    
    # Plot 3: Current angle evolution
    ax3 = axes[1, 0]
    
    ax3.plot(speeds_test, mtpa_angle, 'b-', linewidth=2, label='MTPA Angle')
    ax3.plot(speeds_test, fw_angle, 'r-', linewidth=2, label='Flux Weakening Angle')
    ax3.set_xlabel('Speed (RPM)', fontweight='bold')
    ax3.set_ylabel('Current Angle (degrees)', fontweight='bold')
    ax3.set_title('Current Angle Trajectories', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.axvline(x=base_speed, color='green', linestyle='--', alpha=0.7)
    
    # Plot 4: Performance map overlay
    ax4 = axes[1, 1]
    
    # Create sample efficiency map
    def efficiency_map_model(speed, torque):
        base_eff = 0.85 + 0.15 * np.exp(-((speed - 3000)**2) / (2 * 1500**2))
        torque_factor = 1.0 - 0.3 * np.exp(-((torque - 150)**2) / (2 * 80**2))
        efficiency = base_eff * torque_factor
        efficiency = np.clip(efficiency, 0.6, 0.95)
        return efficiency
    
    efficiency_map = efficiency_map_model(SPEED, TORQUE)
    
    im4 = ax4.contourf(SPEED, TORQUE, efficiency_map, levels=15, cmap='viridis', alpha=0.7)
    
    # Overlay control regions
    ax4.contour(SPEED, TORQUE, region_map, levels=[0.5, 1.5, 2.5], 
               colors=['gray', 'blue', 'red'], linewidths=2, alpha=0.5)
    
    # Add efficiency contours
    efficiency_contours = ax4.contour(SPEED, TORQUE, efficiency_map, 
                                     levels=[0.8, 0.85, 0.9], colors='white', linewidths=1)
    ax4.clabel(efficiency_contours, inline=True, fontsize=8, fmt='%.2f')
    
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Torque (Nm)', fontweight='bold')
    ax4.set_title('Efficiency Map with Control Regions', fontweight='bold')
    
    # Add colorbar
    cbar = plt.colorbar(im4, ax=ax4, label='Efficiency')
    
    plt.tight_layout()
    plt.show()
    
    # Print control strategy insights
    print("⚙️ Control Strategy Insights:")
    print("=" * 40)
    print(f"🎯 Base Speed: {base_speed} RPM (transition between MTPA and flux weakening)")
    print(f"📈 MTPA Region: 0-{base_speed} RPM (optimal torque per ampere)")
    print(f"🚀 Flux Weakening: {base_speed}-6000 RPM (extended speed range)")
    print(f"⚡ Current Management: Dynamic adjustment based on operating conditions")
    print(f"🔋 High Efficiency Region: Concentrated around 3000 RPM, 150 Nm")

visualize_control_strategies()

## Performance Maps

### Types of Performance Maps

#### 1. Efficiency Map (η(N, T))
- **Definition**: Ratio of mechanical output power to electrical input power
- **Range**: Typically 0.6 - 0.95 (60% - 95%)
- **Importance**: Direct impact on vehicle range and energy consumption
- **Units**: Dimensionless (percentage)

#### 2. Power Factor Map (pf(N, T))
- **Definition**: Ratio of real power to apparent power
- **Range**: Typically 0.5 - 1.0
- **Importance**: Affects inverter sizing and grid compatibility
- **Units**: Dimensionless (0 to 1)

#### 3. Loss Maps
- **Copper Losses**: I²R losses in windings
- **Iron Losses**: Hysteresis and eddy current losses in core
- **Stray Losses**: Additional losses not accounted for in main categories

#### 4. Thermal Maps
- **Temperature Distribution**: Heat generation and dissipation
- **Hot Spots**: Areas of concentrated losses
- **Cooling Requirements**: Thermal management design input

In [ ]:
# Performance maps visualization
def visualize_performance_maps():
    """Create comprehensive performance map visualizations"""
    
    # Create sample performance maps
    speed_points = np.linspace(0, 6000, 50)  # RPM
    torque_points = np.linspace(0, 250, 40)   # Nm
    SPEED, TORQUE = np.meshgrid(speed_points, torque_points)
    
    # Generate realistic performance maps
    def generate_efficiency_map(speed, torque):
        """Generate realistic efficiency map"""
        base_eff = 0.85 + 0.15 * np.exp(-((speed - 3000)**2) / (2 * 1500**2))
        torque_factor = 1.0 - 0.3 * np.exp(-((torque - 150)**2) / (2 * 80**2))
        efficiency = base_eff * torque_factor
        
        # Add realistic noise and variation
        efficiency += 0.02 * np.random.randn(*efficiency.shape)
        efficiency = np.clip(efficiency, 0.6, 0.95)
        
        return efficiency
    
    def generate_power_factor_map(efficiency_map):
        """Generate power factor map correlated with efficiency"""
        power_factor = 0.7 + 0.25 * efficiency_map + 0.05 * np.random.randn(*efficiency_map.shape)
        power_factor = np.clip(power_factor, 0.5, 1.0)
        return power_factor
    
    def generate_loss_maps(efficiency_map, power_factor_map):
        """Generate copper and iron loss maps"""
        # Simplified loss model based on operating conditions
        normalized_speed = SPEED / 6000
        normalized_torque = TORQUE / 250
        
        # Copper losses (proportional to current squared)
        copper_loss_base = 500  # Watts
        copper_loss = copper_loss_base * (normalized_torque**2) * (1 + 0.5 * normalized_speed)
        
        # Iron losses (proportional to speed and flux)
        iron_loss_base = 300  # Watts
        iron_loss = iron_loss_base * (normalized_speed**1.5) * (1 + 0.3 * normalized_torque)
        
        return copper_loss, iron_loss
    
    # Generate performance maps
    efficiency_map = generate_efficiency_map(SPEED, TORQUE)
    power_factor_map = generate_power_factor_map(efficiency_map)
    copper_loss_map, iron_loss_map = generate_loss_maps(efficiency_map, power_factor_map)
    total_loss_map = copper_loss_map + iron_loss_map
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Electric Motor Performance Maps', fontsize=16, fontweight='bold')
    
    # Plot 1: Efficiency Map
    ax1 = axes[0, 0]
    im1 = ax1.contourf(SPEED, TORQUE, efficiency_map, levels=20, cmap='viridis')
    ax1.set_xlabel('Speed (RPM)', fontweight='bold')
    ax1.set_ylabel('Torque (Nm)', fontweight='bold')
    ax1.set_title('Efficiency Map', fontsize=14, fontweight='bold')
    plt.colorbar(im1, ax=ax1, label='Efficiency')
    
    # Add high-efficiency contour
    ax1.contour(SPEED, TORQUE, efficiency_map, levels=[0.85, 0.9], colors='white', linewidths=2)
    ax1.clabel(ax1.collections[-1], inline=True, fontsize=10, fmt='%.2f')
    
    # Plot 2: Power Factor Map
    ax2 = axes[0, 1]
    im2 = ax2.contourf(SPEED, TORQUE, power_factor_map, levels=20, cmap='plasma')
    ax2.set_xlabel('Speed (RPM)', fontweight='bold')
    ax2.set_ylabel('Torque (Nm)', fontweight='bold')
    ax2.set_title('Power Factor Map', fontsize=14, fontweight='bold')
    plt.colorbar(im2, ax=ax2, label='Power Factor')
    
    # Plot 3: Combined Performance Index
    ax3 = axes[0, 2]
    performance_index = efficiency_map * power_factor_map  # Combined metric
    im3 = ax3.contourf(SPEED, TORQUE, performance_index, levels=20, cmap='copper')
    ax3.set_xlabel('Speed (RPM)', fontweight='bold')
    ax3.set_ylabel('Torque (Nm)', fontweight='bold')
    ax3.set_title('Performance Index (η × pf)', fontsize=14, fontweight='bold')
    plt.colorbar(im3, ax=ax3, label='Performance Index')
    
    # Plot 4: Copper Loss Map
    ax4 = axes[1, 0]
    im4 = ax4.contourf(SPEED, TORQUE, copper_loss_map, levels=20, cmap='Reds')
    ax4.set_xlabel('Speed (RPM)', fontweight='bold')
    ax4.set_ylabel('Torque (Nm)', fontweight='bold')
    ax4.set_title('Copper Loss Map (W)', fontsize=14, fontweight='bold')
    plt.colorbar(im4, ax=ax4, label='Loss (W)')
    
    # Plot 5: Iron Loss Map
    ax5 = axes[1, 1]
    im5 = ax5.contourf(SPEED, TORQUE, iron_loss_map, levels=20, cmap='Blues')
    ax5.set_xlabel('Speed (RPM)', fontweight='bold')
    ax5.set_ylabel('Torque (Nm)', fontweight='bold')
    ax5.set_title('Iron Loss Map (W)', fontsize=14, fontweight='bold')
    plt.colorbar(im5, ax=ax5, label='Loss (W)')
    
    # Plot 6: Total Loss Map
    ax6 = axes[1, 2]
    im6 = ax6.contourf(SPEED, TORQUE, total_loss_map, levels=20, cmap='hot')
    ax6.set_xlabel('Speed (RPM)', fontweight='bold')
    ax6.set_ylabel('Torque (Nm)', fontweight='bold')
    ax6.set_title('Total Loss Map (W)', fontsize=14, fontweight='bold')
    plt.colorbar(im6, ax=ax6, label='Total Loss (W)')
    
    plt.tight_layout()
    plt.show()
    
    # Calculate and print performance metrics
    high_eff_threshold = 0.85
    high_pf_threshold = 0.85
    
    high_eff_area = np.sum(efficiency_map > high_eff_threshold) / efficiency_map.size * 100
    high_pf_area = np.sum(power_factor_map > high_pf_threshold) / power_factor_map.size * 100
    
    peak_efficiency = np.max(efficiency_map)
    peak_power_factor = np.max(power_factor_map)
    peak_eff_loc = np.unravel_index(np.argmax(efficiency_map), efficiency_map.shape)
    peak_pf_loc = np.unravel_index(np.argmax(power_factor_map), power_factor_map.shape)
    
    print("📊 Performance Map Analysis:")
    print("=" * 40)
    print(f"⚡ Peak Efficiency: {peak_efficiency:.3f} at {SPEED[peak_eff_loc[1], peak_eff_loc[0]]:.0f} RPM, {TORQUE[peak_eff_loc[1], peak_eff_loc[0]]:.0f} Nm")
    print(f"🔋 Peak Power Factor: {peak_power_factor:.3f} at {SPEED[peak_pf_loc[1], peak_pf_loc[0]]:.0f} RPM, {TORQUE[peak_pf_loc[1], peak_pf_loc[0]]:.0f} Nm")
    print(f"🎯 High-Efficiency Area (>85%): {high_eff_area:.1f}% of operating region")
    print(f"⚡ High-Power Factor Area (>85%): {high_pf_area:.1f}% of operating region")
    print(f"🔥 Peak Total Loss: {np.max(total_loss_map):.1f} W")
    print(f"💡 Optimal Operating Region: ~2500-3500 RPM, 120-180 Nm")

visualize_performance_maps()

## Design Parameter Sensitivity

### Key Design Parameters

#### IPM Motor Parameters (X₁-X₁₂)
- **X₁-X₄**: Magnet dimensions and positioning
- **X₅-X₈**: Rotor geometry parameters
- **X₉-X₁₁**: Stator winding configuration
- **X₁₂**: Air gap dimension

#### FSCW Motor Parameters (X₁-X₁₀)
- **X₁-X₄**: Stator slot dimensions
- **X₅**: Air gap dimension
- **X₆-X₉**: Rotor geometry parameters
- **X₁₀**: Magnet thickness

### Sensitivity Analysis

Understanding how each parameter affects performance is crucial for:
- **Design Optimization**: Focusing on high-impact parameters
  
- **Manufacturing Tolerances**: Setting realistic tolerance requirements
- **Cost-Benefit Analysis**: Balancing performance improvements against cost
- **Design Space Exploration**: Efficient sampling of parameter combinations

In [ ]:
# Design parameter sensitivity analysis
def analyze_parameter_sensitivity():
    """Analyze sensitivity of motor design parameters"""
    
    # Define parameter ranges for sensitivity analysis
    ipm_params = {
        'Magnet Width': {'range': (15, 25), 'sensitivity': 0.8},
        'Air Gap': {'range': (0.5, 1.5), 'sensitivity': 0.6},
        'Slot Depth': {'range': (20, 40), 'sensitivity': 0.4},
        'Core Length': {'range': (80, 120), 'sensitivity': 0.7},
        'Magnet Height': {'range': (3, 8), 'sensitivity': 0.6}
    }
    
    fscw_params = {
        'Slot Width': {'range': (8, 18), 'sensitivity': 0.6},
        'Air Gap': {'range': (0.5, 1.5), 'sensitivity': 0.8},
        'Slot Depth': {'range': (15, 35), 'sensitivity': 0.7},
        'Core Length': {'range': (90, 130), 'sensitivity': 0.5},
        'Magnet Thickness': {'range': (3, 7), 'sensitivity': 0.6}
    }
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Motor Design Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: IPM parameter sensitivity
    ax1 = axes[0, 0]
    
    params_ipm = list(ipm_params.keys())
    sensitivity_ipm = [ipm_params[param]['sensitivity'] for param in params_ipm]
    
    bars1 = ax1.barh(params_ipm, sensitivity_ipm, color='#3498DB', alpha=0.7)
    ax1.set_xlabel('Sensitivity Score', fontweight='bold')
    ax1.set_title('IPM Motor Parameter Sensitivity', fontsize=14, fontweight='bold')
    ax1.set_xlim(0, 1)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, sensitivity) in enumerate(zip(bars1, sensitivity_ipm)):
        ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{sensitivity:.2f}', ha='left', va='center', fontweight='bold')
    
    # Plot 2: FSCW parameter sensitivity
    ax2 = axes[0, 1]
    
    params_fscw = list(fscw_params.keys())
    sensitivity_fscw = [fscw_params[param]['sensitivity'] for param in params_fscw]
    
    bars2 = ax2.barh(params_fscw, sensitivity_fscw, color='#E74C3C', alpha=0.7)
    ax2.set_xlabel('Sensitivity Score', fontweight='bold')
    ax2.set_title('FSCW Motor Parameter Sensitivity', fontsize=14, fontweight='bold')
    ax2.set_xlim(0, 1)
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, sensitivity) in enumerate(zip(bars2, sensitivity_fscw)):
        ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{sensitivity:.2f}', ha='left', va='center', fontweight='bold')
    
    # Plot 3: Parameter impact comparison
    ax3 = axes[1, 0]
    
    all_params = params_ipm + params_fscw
    all_sensitivity = sensitivity_ipm + sensitivity_fscw
    motor_types = ['IPM'] * len(params_ipm) + ['FSCW'] * len(params_fscw)
    
    x_pos = np.arange(len(all_params))
    width = 0.35
    
    bars_ipm = ax3.bar(x_pos[:len(params_ipm)] - width/2, sensitivity_ipm, width, 
                        label='IPM', color='#3498DB', alpha=0.7)
    bars_fscw = ax3.bar(x_pos[len(params_ipm):] + width/2, sensitivity_fscw, width, 
                         label='FSCW', color='#E74C3C', alpha=0.7)
    
    ax3.set_xlabel('Design Parameters', fontweight='bold')
    ax3.set_ylabel('Sensitivity Score', fontweight='bold')
    ax3.set_title('Parameter Impact Comparison', fontsize=14, fontweight='bold')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(all_params, rotation=45, ha='right')
    ax3.legend()
    ax3.set_ylim(0, 1)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Design optimization insights
    ax4 = axes[1, 1]
    
    # Create scatter plot showing parameter ranges vs sensitivity
    param_ranges_ipm = [ipm_params[param]['range'][1] - ipm_params[param]['range'][0] 
                       for param in params_ipm]
    param_ranges_fscw = [fscw_params[param]['range'][1] - fscw_params[param]['range'][0] 
                         for param in params_fscw]
    
    ax4.scatter(param_ranges_ipm, sensitivity_ipm, s=100, alpha=0.7, 
               color='#3498DB', label='IPM', edgecolors='black')
    ax4.scatter(param_ranges_fscw, sensitivity_fscw, s=100, alpha=0.7, 
               color='#E74C3C', label='FSCW', edgecolors='black')
    
    # Add annotations for high-impact parameters
    high_impact_threshold = 0.7
    
    for i, (param, sensitivity, range_val) in enumerate(zip(params_ipm, sensitivity_ipm, param_ranges_ipm)):
        if sensitivity > high_impact_threshold:
            ax4.annotate(f'{param}\n({sensitivity:.2f})', 
                        (range_val, sensitivity), 
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.7))
    
    for i, (param, sensitivity, range_val) in enumerate(zip(params_fscw, sensitivity_fscw, param_ranges_fscw)):
        if sensitivity > high_impact_threshold:
            ax4.annotate(f'{param}\n({sensitivity:.2f})', 
                        (range_val, sensitivity), 
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.7))
    
    ax4.set_xlabel('Parameter Range (mm)', fontweight='bold')
    ax4.set_ylabel('Sensitivity Score', fontweight='bold')
    ax4.set_title('Design Range vs Impact Analysis', fontsize=14, fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print sensitivity analysis summary
    print("🔍 Design Parameter Sensitivity Analysis:")
    print("=" * 50)
    
    print("\n📊 IPM Motor - High Impact Parameters:")
    high_impact_ipm = [(param, info) for param, info in ipm_params.items() if info['sensitivity'] > 0.7]
    for param, info in sorted(high_impact_ipm, key=lambda x: x[1]['sensitivity'], reverse=True):
        print(f"   🔹 {param}: Sensitivity = {info['sensitivity']:.2f}, Range = {info['range']} mm")
    
    print("\n📊 FSCW Motor - High Impact Parameters:")
    high_impact_fscw = [(param, info) for param, info in fscw_params.items() if info['sensitivity'] > 0.7]
    for param, info in sorted(high_impact_fscw, key=lambda x: x[1]['sensitivity'], reverse=True):
        print(f"   🔹 {param}: Sensitivity = {info['sensitivity']:.2f}, Range = {info['range']} mm")
    
    print("\n💡 Design Optimization Insights:")
    print(f"   🎯 Focus on high-sensitivity parameters for maximum performance improvement")
    print(f"   ⚖️ Air gap is critical for both motor types (sensitivity > 0.6)")
    print(f"   📏 Parameter ranges vary significantly between motor types")
    print(f"   🔧 Magnet dimensions have different optimization priorities")

analyze_parameter_sensitivity()

## Section Summary

### 🎯 Key Takeaways from Motor Fundamentals

1. **Motor Type Selection**:
   - **IPM Motors**: Better for high-torque, high-speed applications
   - **FSCW Motors**: Better for efficiency, low-cogging applications
   - **Trade-offs**: Consider application requirements when selecting motor type

2. **Control Strategy Understanding**:
   - **MTPA**: Optimal below base speed for maximum efficiency
   - **Flux Weakening**: Extends speed range above base speed
   - **Dynamic Control**: Adjust parameters based on operating conditions

3. **Performance Map Characteristics**:
   - **Efficiency Maps**: Critical for range prediction and energy consumption
   - **Power Factor Maps**: Important for inverter sizing and compatibility
   - **Loss Maps**: Essential for thermal management design
   - **Operating Envelopes**: Define feasible operating regions

4. **Design Parameter Impact**:
   - **High-Impact Parameters**: Air gap, magnet dimensions, core length
   - **Sensitivity Analysis**: Focus optimization efforts on high-impact parameters
   - **Manufacturing Tolerances**: Balance performance improvements against cost

### 🔧 Practical Applications

- **Vehicle Design**: Select appropriate motor type for specific vehicle requirements
- **Control System Development**: Implement MTPA and flux weakening algorithms
- **Performance Prediction**: Use maps to estimate vehicle range and efficiency
- **Thermal Management**: Design cooling systems based on loss map analysis
- **Design Optimization**: Focus on high-sensitivity parameters for maximum improvement

This foundation provides the necessary context for understanding how deep learning models can predict these complex performance characteristics from motor geometry and operating conditions.